# Day 4 Lab — End-to-End Alignment Loop with a Small LM

**Goal:** use the same small pretrained LM to *generate* outputs, *score* them with a simple reward, *tune* on improved targets, and *re-evaluate* — and actually watch **training loss go down** and **reward go up** while generations get cleaner.

This is a minimal CPU capstone that turns telemetry/reward ideas into a concrete model-improvement loop.

**What makes the loop actually work here (the four fixes):**
1. **Score the completion, not the prompt.** The reward is computed on the *generated* text only. (The prompt already contains every `must_include` term, so scoring prompt+completion made reward a constant `1.0`.)
2. **Prompt the Instruct model with its chat template** so it produces clean, grounded answers instead of rambling.
3. **Greedy decoding with no repetition penalty.** A repetition penalty over the full context *suppresses copying the grounded facts* (dates, owners) straight out of the context, which is exactly what we want the model to do.
4. **Tune the top transformer blocks + LM head on the improved targets** (with an EOS so the model learns to stop). Tuning only the tied embedding was too weak and unstable; this converges smoothly and cleanly.


## Notebook flow (what actually happens, step by step)

**Important distinction:** this notebook does **not** train a separate "judge" model (that's the day6 notebook). Here, the *same* small LM plays two roles at once — it generates answers, and a plain Python `reward()` function (not a model) scores them. The loop below only ever tunes the **answer-generating model**, using that reward as a diagnostic, not as a training signal (no RL/PPO is happening — the actual tuning step is supervised fine-tuning on hand-written `target` answers).

```mermaid
flowchart TD
    A["Cell 2: Setup\nload_small_lm, prompt builders,\nsft_loss, make_tiny_trainable,\nrun_sft_steps"] --> B["Cell 4: Define tasks\n3 labeled tasks with\ninstruction + context + target\n+ reward() scoring function"]
    B --> C["Cell 6: BEFORE eval\ngenerate_completion() for each task\nreward() scores each output\navg reward recorded as 'before'"]
    C --> D["Cell 8: Fine-tune\nrun_sft_steps() freezes most\nweights, trains only top transformer\nblocks + LM head on the 3 targets"]
    D --> E["Cell 10: AFTER eval\nSAME generate_completion + reward()\ncalled again -> avg reward 'after'\ncompared against 'before'"]
    E --> F["Cell 12: Generalization check\nheld-out new_task (never trained on)\ngenerated + scored to see if the\nmodel generalized or just memorized"]
```

**Where each piece of "the loop" lives:**
1. **Generate** — `generate_completion()` (Cell 2) produces the model's current answer for a task.
2. **Score** — `reward()` (Cell 4) checks the answer against `must_include` facts and penalizes verbosity/hedging. This is the automatic scorer standing in for a human/judge grade.
3. **Tune** — `run_sft_steps()` (Cell 2, called in Cell 8) updates model weights so its answers move closer to the hand-written `target` strings.
4. **Re-evaluate** — Cell 10 reruns generate+score and compares `before` vs `after` reward, and training loss (`losses[0]` vs `losses[-1]`).
5. **Stress-test generalization** — Cell 12 runs the tuned model on a task it never saw, to check whether it truly learned the *pattern* (grounded, concise answers) or just memorized the 3 training examples.


## Cheat sheet: the 6-block skeleton (memorize this, not the code)

Every small-LM tuning notebook in this bootcamp (day6 judge, day7 alignment loop) is the **same 6 blocks** wearing different clothes. Internalize this shape first — the exact code is just a lookup once the shape is automatic.

| # | Block | One-sentence "why" | Function(s) here |
|---|-------|---------------------|-------------------|
| 1 | **Load model** | Get a small model + tokenizer onto CPU, with a safe fallback if download fails. | `load_small_lm()` |
| 2 | **Freeze most weights** | Full fine-tuning is slow/unstable on CPU with tiny data; unfreeze only a small slice (top blocks + head) so training is cheap and less prone to overfitting. | `make_tiny_trainable()` |
| 3 | **Compute masked loss** | Build prompt+target, mask the prompt with `-100` so the model is only scored on generating the *answer* tokens, not re-predicting context it was already given. | `sft_loss()`, `build_full_inputs()` |
| 4 | **Train loop** | Repeatedly: forward pass → loss → `backward()` (gradient = which way reduces loss) → `opt.step()` (nudge weights that direction). Do this for several epochs so loss trends down. | `run_sft_steps()` |
| 5 | **Eval before/after** | Generate completions with the *same* function/prompts both times, score them the same way, so any difference is attributable to tuning — not confounded by prompt/format changes. | `generate_completion()`, `run_eval()` / `eval_judge()` |
| 6 | **Compare + stress-test** | Look at loss trend (should ↓) and score/reward trend (should ↑). Then test on a held-out example never trained on, to catch memorization vs. real generalization. | Section 4 summary, Section 5 `new_task` |

**Trigger questions — answer these from memory, no code, before you look anything up:**
- Why mask the prompt tokens with `-100` instead of just training on the whole sequence?
- Why freeze most of the model instead of fine-tuning everything?
- What's the difference between generating a completion (`model.eval()` + `no_grad()`) and training (`model.train()` + `backward()`)?
- Why must the "before" and "after" evaluation use the exact same prompts/scoring function?
- Why does a held-out example (never trained on) matter, even if "before → after" reward looks great?

**Drill routine:** Day 1 — recite the 6 block names only. Day 2 — recite the "why" column only. Day 3 — pick one block, rebuild it blank-page with your own variable names, then diff against the real function. Repeat for the next block only once the previous one feels automatic.


In [3]:
# If needed, uncomment the install line below in a fresh notebook environment.
# %pip install -q "transformers>=4.41" "accelerate>=0.30" torch

import math, random, re, json, time
from pprint import pprint
import torch
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer

random.seed(7)
torch.manual_seed(7)
DEVICE = "cpu"
# Keep the default very small for CPU. Swap to Qwen/Qwen2.5-0.5B-Instruct if your laptop has enough RAM.
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
FALLBACK_MODEL = "sshleifer/tiny-gpt2"

# A short system prompt keeps answers compact and grounded (used when the model has a chat template).
SYSTEM_PROMPT = (
    "You are a precise enterprise assistant. Answer in one or two short, grounded "
    "sentences using only the facts in the context. Do not speculate."
)

def load_small_lm(model_name=MODEL_NAME):
    """Load the tokenizer + model this whole lab will generate from, score, and tune."""
    # STEP 1: Try to load the preferred small model; fall back to an even tinier model
    # if there's no network access or the model fails to download.
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name)
        loaded = model_name
    except Exception as exc:
        print(f"Could not load {model_name}: {exc}\nFalling back to {FALLBACK_MODEL}.")
        tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL)
        model = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL)
        loaded = FALLBACK_MODEL
    # STEP 2: Some small models ship without a pad token; reuse EOS so tokenization doesn't crash.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # STEP 3: Untie the LM head from the input embeddings. With tied weights every backward pass
    # must travel all the way back to the embeddings (slow) and destabilizes generation;
    # untying lets us cheaply fine-tune just the output projection + top blocks.
    if getattr(model.config, "tie_word_embeddings", False) and hasattr(model, "lm_head"):
        model.lm_head.weight = torch.nn.Parameter(model.lm_head.weight.detach().clone())
        model.config.tie_word_embeddings = False
    model.to(DEVICE)
    model.eval()  # start in eval mode; run_sft_steps() will flip to train() only while tuning.
    print("Loaded:", loaded)
    return tokenizer, model

def task_prompt(x):
    """Plain-text fallback prompt format (used only when the tokenizer has no chat template)."""
    return f"Instruction: {x['instruction']}\nContext: {x['context']}\nAnswer: "

def chat_messages(x):
    """Build the chat-style message list (system + user turn) for instruct-tuned models."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{x['instruction']}\nContext: {x['context']}"},
    ]

def _has_chat_template(tokenizer):
    """True if this tokenizer knows how to format chat messages (instruct models do)."""
    return getattr(tokenizer, "chat_template", None) is not None

def _eos_ids(tokenizer):
    """Collect every token id that should count as 'end of generation' for this model."""
    ids = []
    if tokenizer.eos_token_id is not None:
        ids.append(tokenizer.eos_token_id)
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end, int) and im_end >= 0 and im_end not in ids:
        ids.append(im_end)  # chat models end an assistant turn with <|im_end|>
    return ids or None

def build_prompt_inputs(tokenizer, x):
    """Token ids for the prompt only (ready for generation)."""
    # STEP 1: Prefer the model's own chat template if it has one -- this reproduces the
    # exact formatting (system/user/assistant markers) the instruct model was trained on.
    if _has_chat_template(tokenizer):
        return tokenizer.apply_chat_template(
            chat_messages(x), add_generation_prompt=True,
            return_dict=True, return_tensors="pt").to(DEVICE)
    # STEP 2: Otherwise fall back to a plain text prompt.
    return tokenizer(task_prompt(x), return_tensors="pt",
                     truncation=True, max_length=256).to(DEVICE)

def build_full_inputs(tokenizer, x):
    """Token ids for prompt + target (+ end token), used for supervised fine-tuning."""
    # STEP 1: Same chat-template-or-plain-text choice as above, but this time we also append
    # the assistant's target answer (plus an EOS token) so the model can learn to produce it
    # AND learn to stop afterward.
    if _has_chat_template(tokenizer):
        msgs = chat_messages(x) + [{"role": "assistant", "content": x["target"]}]
        return tokenizer.apply_chat_template(
            msgs, return_dict=True, return_tensors="pt").to(DEVICE)
    text = task_prompt(x) + x["target"] + (tokenizer.eos_token or "")
    return tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)

def generate_completion(tokenizer, model, x, max_new_tokens=48):
    """Greedy-decode and return ONLY the newly generated completion (not the prompt echo)."""
    model.eval()  # STEP 1: eval mode -- no dropout, deterministic generation.
    inputs = build_prompt_inputs(tokenizer, x)
    prompt_len = inputs["input_ids"].shape[1]  # STEP 2: remember where the prompt ends.
    with torch.no_grad():
        # STEP 3: Greedy decoding (do_sample=False), no repetition penalty -- this lets the
        # model copy grounded facts (dates, owners) straight out of the context instead of
        # being discouraged from repeating them.
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=_eos_ids(tokenizer),
        )
    # STEP 4: Slice off everything up to prompt_len so we only keep the newly generated tokens,
    # then decode them into text.
    gen_ids = out[0][prompt_len:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

def sft_loss(tokenizer, model, x, max_length=256):
    """Compute the supervised fine-tuning loss for one task row (answer tokens only)."""
    # STEP 1: Build the full (prompt + target) sequence and the prompt-only sequence.
    full = build_full_inputs(tokenizer, x)
    prompt = build_prompt_inputs(tokenizer, x)

    # STEP 2: Copy the full input ids to use as labels, then mask out the prompt portion
    # with -100 so the loss only scores the model on generating the target answer tokens
    # (we don't want to "train" it to re-predict the context it was already given).
    labels = full["input_ids"].clone()
    prompt_len = min(prompt["input_ids"].shape[1], labels.shape[1])
    labels[:, :prompt_len] = -100

    # STEP 3: Forward pass -- the model computes cross-entropy loss internally when `labels` is given.
    outputs = model(input_ids=full["input_ids"],
                    attention_mask=full.get("attention_mask"),
                    labels=labels)
    return outputs.loss

def make_tiny_trainable(model, top_blocks=4):
    """Unfreeze only the top-K transformer blocks + final norm + LM head.
    Lower layers and the token embeddings stay frozen, so each backward pass only
    traverses the top blocks (cheap on CPU) while still giving the model enough
    capacity to route each distinct prompt to its own grounded answer."""
    # STEP 1: Figure out how many transformer blocks/layers this model has by scanning
    # parameter names for a "...layers.N..." or "...h.N..." pattern.
    layer_ids = set()
    for name, _ in model.named_parameters():
        m = re.search(r"\.(?:layers|h)\.(\d+)\.", name)
        if m:
            layer_ids.add(int(m.group(1)))
    n_layers = (max(layer_ids) + 1) if layer_ids else 0
    # STEP 2: Decide which layer indices count as "top" (closest to the output).
    keep = set(range(max(0, n_layers - top_blocks), n_layers))

    # STEP 3: Freeze everything first...
    for p in model.parameters():
        p.requires_grad = False
    trainable = []
    # STEP 4: ...then selectively unfreeze: the LM head, the final normalization layer,
    # and any parameter belonging to one of the "top" transformer blocks.
    for name, p in model.named_parameters():
        m = re.search(r"\.(?:layers|h)\.(\d+)\.", name)
        in_block = m is not None
        is_top_block = in_block and int(m.group(1)) in keep
        is_head = "lm_head" in name
        is_final_norm = (not in_block) and (("norm" in name) or ("ln_f" in name))
        if is_head or is_final_norm or is_top_block:
            p.requires_grad = True
            trainable.append(name)
    # STEP 5: Safety net -- if the naming pattern didn't match anything, just unfreeze
    # the last few tensors so training still has something to update.
    if not trainable:  # generic fallback
        for name, p in list(model.named_parameters())[-8:]:
            p.requires_grad = True
            trainable.append(name)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable tensors: {len(trainable)} | trainable params: {n_params:,} "
          f"| top {top_blocks} blocks {sorted(keep)}")
    return trainable

def run_sft_steps(tokenizer, model, rows, epochs=8, lr=5e-4, top_blocks=4):
    """One epoch = accumulate the (answer-only) loss over every row, then step.
    Averaging over the small, fixed batch gives a smooth, monotonically decreasing loss."""
    # STEP 1: Freeze most of the model, leaving only the top blocks + head trainable.
    make_tiny_trainable(model, top_blocks=top_blocks)
    # STEP 2: Optimizer only ever sees the trainable (unfrozen) parameters.
    opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    losses = []
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        epoch_loss = 0.0
        # STEP 3: Accumulate gradients across every row in this epoch before stepping once --
        # this is what makes the loss curve smooth instead of noisy/oscillating.
        for x in rows:
            loss = sft_loss(tokenizer, model, x) / len(rows)  # divide so the batch acts like an average
            loss.backward()
            epoch_loss += float(loss.detach())
        # STEP 4: Apply one optimizer update per epoch, using the accumulated gradients.
        opt.step()
        losses.append(round(epoch_loss, 4))
        print(f"epoch {ep:02d} | loss {epoch_loss:.4f}")
    print("epoch losses:", losses)
    return losses


## 1. Existing production-style examples
These reuse the support, incident, and tool-use themes already in the deck/labs. Each task carries the grounded `target` answer we want the model to learn, plus the `must_include` terms the reward checks for.

In [4]:
# STEP 1: Define the task set. Each row bundles an instruction + context (what the model sees),
# a list of `must_include` facts the answer should mention, and a grounded `target` answer
# (used as the supervised fine-tuning label in Section 3).
tasks = [
    {"id":"refund", "instruction":"Summarize refund status for a support manager.",
     "context":"Refund approved Jun 5. Expected Jun 10. Owner: Billing.",
     "must_include":["Jun 5", "Jun 10", "Billing"],
     "target":"Refund approved Jun 5; expected Jun 10. Owner: Billing."},
    {"id":"incident", "instruction":"Brief this incident for an engineering manager.",
     "context":"Impact: 21 tenants. Cause: token refresh bug. Owner: Auth. Next action: hotfix.",
     "must_include":["21", "token", "Auth", "hotfix"],
     "target":"21 tenants are impacted by a token refresh bug. Auth owns the hotfix next action."},
    {"id":"tool", "instruction":"State the next tool action for this task.",
     "context":"User asks for recent unread emails. Required source: mailbox search. Constraint: do not search chats.",
     "must_include":["email", "mailbox", "unread"],
     "target":"Search unread emails in the mailbox and summarize only the relevant email results."},
]

def reward(output, x):
    """Score a generated `output` string against task `x` -- higher is better (max 1.0)."""
    # STEP 1: Coverage -- fraction of the must_include terms that actually appear in the output.
    lower = output.lower()
    coverage = sum(term.lower() in lower for term in x["must_include"]) / len(x["must_include"])
    # STEP 2: Penalties -- subtract points for rambling (>80 words) or hedging language
    # ("probably"/"maybe"), since a grounded, concise answer shouldn't need either.
    verbose_penalty = 0.2 if len(output.split()) > 80 else 0.0
    unsupported_penalty = 0.2 if "probably" in lower or "maybe" in lower else 0.0
    # STEP 3: Final reward is coverage minus penalties, clipped at 0 so it never goes negative.
    return round(max(0, coverage - verbose_penalty - unsupported_penalty), 2)

# The rows we fine-tune on are just the tasks themselves (prompt built from instruction+context,
# target is the grounded answer). This mimics an alignment loop where low-reward outputs are
# patched with better supervised data.
train_rows = tasks
pprint({"prompt": task_prompt(tasks[0]), "target": tasks[0]["target"]})


{'prompt': 'Instruction: Summarize refund status for a support manager.\n'
           'Context: Refund approved Jun 5. Expected Jun 10. Owner: Billing.\n'
           'Answer: ',
 'target': 'Refund approved Jun 5; expected Jun 10. Owner: Billing.'}


## 2. Generate and score **before** tuning
Note the reward is computed on `out` — the **completion only** — so it reflects what the model actually generates, not the prompt we handed it.

In [ ]:
# STEP 1: Load the (untuned) model -- this is the "before" state for the whole alignment loop.
tokenizer, model = load_small_lm()

def run_eval(tokenizer, model, title):
    """Generate a completion for every task, score it with `reward`, and report the average."""
    rows = []
    for x in tasks:
        # STEP 1: Generate the model's completion for this task (prompt only, no target shown).
        out = generate_completion(tokenizer, model, x, max_new_tokens=48)
        # STEP 2: Score the COMPLETION (not prompt+completion) against this task's must_include terms.
        score = reward(out, x)                      # score the COMPLETION, not prompt+completion
        rows.append({"id": x["id"], "score": score, "output": out})
    # STEP 3: Average reward across all tasks -- this is the single number we track before/after tuning.
    avg = round(sum(r["score"] for r in rows) / len(rows), 2)
    print("\n", title, "avg reward", avg)
    for r in rows:
        print("\n---", r["id"], "score", r["score"], "---")
        print(r["output"])
    return rows

# STEP 2: Run the baseline (pre-tuning) evaluation -- our reference point for comparison later.
before = run_eval(tokenizer, model, "before")


Loaded: HuggingFaceTB/SmolLM2-135M-Instruct

 before avg reward 0.81

--- refund score 0.67 ---
A refund for a support manager was approved on Jun 5. The refund was expected on Jun 10.

--- incident score 0.75 ---
A 21-tenant tenant was impacted by a token refresh bug, causing a temporary outage. The tenant was hotfixed, and the issue was resolved.

--- tool score 1.0 ---
User asks for recent unread emails.

The next tool action for this task is to search for unread emails in the mailbox search.


## 3. Tune on improved targets
We fine-tune the **top transformer blocks + LM head** on the grounded targets (answer tokens only). Watch the per-epoch loss fall smoothly — averaging the loss over the small fixed batch each epoch is what keeps it monotonic (the original single-example-per-step loop just oscillated).

On a modern laptop CPU this cell takes a few minutes (each epoch runs a full forward pass over every example).

In [6]:
# STEP 1: Run the tuning loop -- 8 epochs, gradients accumulated over all 3 tasks each epoch,
# only the top 4 transformer blocks + LM head are trainable. Watch `losses` fall smoothly.
losses = run_sft_steps(tokenizer, model, train_rows, epochs=3, lr=5e-4, top_blocks=4)


Trainable tensors: 38 | trainable params: 42,472,512 | top 4 blocks [26, 27, 28, 29]
epoch 00 | loss 1.9641
epoch 01 | loss 0.8324
epoch 02 | loss 0.2716
epoch losses: [1.9641, 0.8324, 0.2716]


## 4. Generate and score **after** tuning
Same eval, same reward. The completions should now match the grounded targets and stop cleanly, and the average reward should climb.

In [7]:
# STEP 1: Re-run the identical eval (same tasks, same reward function) now that the model is tuned.
after = run_eval(tokenizer, model, "after")

# STEP 2: Summarize the whole loop -- did training loss go down AND did reward go up?
# These two checks together are the "did the alignment loop actually work" signal.
avg = lambda rows: round(sum(r["score"] for r in rows) / len(rows), 2)
print("\n=== Alignment loop summary ===")
print(f"train loss : {losses[0]:.3f} -> {losses[-1]:.3f}  (decreased: {losses[-1] < losses[0]})")
print(f"avg reward : {avg(before)} -> {avg(after)}  (increased: {avg(after) >= avg(before)})")



 after avg reward 1.0

--- refund score 1.0 ---
Refund approved Jun 5; expected Jun 10. Owner: Billing.

--- incident score 1.0 ---
21 tenants are impacted by a token refresh bug. Auth owns the hotfix next action.

--- tool score 1.0 ---
Search unread emails in the mailbox and summarize only the relevant email results.

=== Alignment loop summary ===
train loss : 1.964 -> 0.272  (decreased: True)
avg reward : 0.81 -> 1.0  (increased: True)


## 5. Add a new slice
Add one more task below, run eval, and inspect whether tuning generalized or only memorized the examples. The trained model usually produces a coherent, partially-grounded answer here — good fodder for the discussion.

In [8]:
# STEP 1: Define a held-out task the model was never tuned on -- this tests generalization
# vs. memorization of the 3 training examples.
new_task = {"id":"new-routing", "instruction":"Route this case to the right owner.",
    "context":"Customer cannot access invoice. Root area: Billing portal. Owner: Billing Apps.",
    "must_include":["Billing", "invoice"],
    "target":"Route to Billing Apps because the issue is invoice access in the Billing portal."}

# STEP 2: Generate + score this new task with the same reward function used above, and inspect
# whether the model produces a coherent, grounded answer or just imitates training-set phrasing.
out = generate_completion(tokenizer, model, new_task, max_new_tokens=48)
print("reward:", reward(out, new_task))
print(out)


reward: 0.5
Search for the Billing Portal and navigate to the root area.


## Discussion
- **Did reward improve globally or only for memorized examples?** On the three tuned slices the completions converge to the grounded targets (reward → 1.0). The held-out *new-routing* slice improves in style but only partially covers its `must_include` terms — evidence of memorization outweighing generalization with just three examples.
- **Which reward terms are likely to be gamed?** `coverage` rewards containing the keywords, so a model can score well by stuffing must-include terms without being faithful; the verbosity/hedging penalties are easy to dodge and say nothing about correctness.
- **Why did scoring the *completion* matter?** The prompt already contains every `must_include` term, so scoring prompt+completion pinned reward at 1.0 and hid all signal. Always score what the model *generated*.
- **What offline eval gate would you require before a canary rollout?** A held-out slice suite (not the training examples), a faithfulness/grounding check beyond keyword coverage, and a regression bar (no drop vs. the current production model) before any canary.